# 01. 데이터 로딩과 품질 점검

## 목표

Adult Census 원본을 Pandas와 Polars로 각각 읽고 행·열, 결측치, 중복 행이 일치하는지 확인합니다.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if not (project_root / 'src').exists():
    raise RuntimeError('adult-marriage-education-analysis 프로젝트 루트에서 실행하세요.')
sys.path.insert(0, str(project_root))
project_root

PosixPath('/Users/beeyaaa/Workspace/SK-SKALA/day15_data/adult-marriage-education-analysis')

In [2]:
from src.data import load_pandas, load_polars, compare_loaders, resolve_raw_path

raw_path = resolve_raw_path(project_root)
pandas_df = load_pandas(raw_path)
polars_df = load_polars(raw_path)
print('원본 경로:', raw_path)
print('Pandas:', pandas_df.shape)
print('Polars:', polars_df.shape)

원본 경로: /Users/beeyaaa/Workspace/SK-SKALA/day15_data/adult-marriage-analysis/data/raw/adult.data
Pandas: (32561, 15)
Polars: (32561, 15)


## 원본 데이터 미리보기

In [3]:
pandas_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
polars_df.head()

age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str
39,"""State-gov""",77516,"""Bachelors""",13,"""Never-married""","""Adm-clerical""","""Not-in-family""","""White""","""Male""",2174,0,40,"""United-States""","""<=50K"""
50,"""Self-emp-not-inc""",83311,"""Bachelors""",13,"""Married-civ-spouse""","""Exec-managerial""","""Husband""","""White""","""Male""",0,0,13,"""United-States""","""<=50K"""
38,"""Private""",215646,"""HS-grad""",9,"""Divorced""","""Handlers-cleaners""","""Not-in-family""","""White""","""Male""",0,0,40,"""United-States""","""<=50K"""
53,"""Private""",234721,"""11th""",7,"""Married-civ-spouse""","""Handlers-cleaners""","""Husband""","""Black""","""Male""",0,0,40,"""United-States""","""<=50K"""
28,"""Private""",338409,"""Bachelors""",13,"""Married-civ-spouse""","""Prof-specialty""","""Wife""","""Black""","""Female""",0,0,40,"""Cuba""","""<=50K"""


## Pandas와 Polars 비교

In [5]:
loader_comparison = compare_loaders(pandas_df, polars_df)
loader_comparison.to_csv(project_root / 'reports' / 'tables' / 'loader_comparison.csv', index=False)
loader_comparison

,항목,Pandas,Polars,일치 여부
0,행 수,32561,32561,True
1,열 수,15,15,True
2,전체 결측치,4262,4262,True
3,완전 중복 행,24,24,True


## 결측치와 중복 행

In [6]:
missing_summary = (
    pandas_df.isna().sum().rename('결측치 수').to_frame()
    .assign(**{'결측률(%)': lambda x: x['결측치 수'] / len(pandas_df) * 100})
    .query('`결측치 수` > 0')
    .sort_values('결측치 수', ascending=False)
)
print('완전 중복 행:', pandas_df.duplicated().sum())
missing_summary

완전 중복 행: 24


,결측치 수,결측률(%)
occupation,1843,5.660146
workclass,1836,5.638647
native-country,583,1.790486


## 정리

- 두 라이브러리 결과가 일치하는지 확인합니다.
- 개인 식별자가 없어 완전 중복 행은 자동 삭제하지 않습니다.
- 범주형 결측치는 이후 Pipeline 내부에서 처리합니다.

In [7]:
assert loader_comparison['일치 여부'].all()
assert len(pandas_df) == 32_561
print('검증 완료: 데이터 로딩과 품질 점검이 정상입니다.')

검증 완료: 데이터 로딩과 품질 점검이 정상입니다.
